In [6]:
import gurobipy as gp
from gurobipy import GRB, quicksum
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

# ------------------------
# Load Data
# ------------------------
accommodation_path = r'C:/Users/Felicia/OneDrive - Nanyang Technological University/School/BC2411 Prescriptive Analytics From Data To Decision/Assignments & Projects/accommodation_data.csv'
attractions_path = r'C:/Users/Felicia/Downloads/amsterdam_named_attractions.csv'

df_airbnb = pd.read_csv(accommodation_path)
df_attractions = pd.read_csv(attractions_path)

# Extract Airbnb info
airbnb_names = df_airbnb.iloc[:, 0].values  # Airbnb names from the first column
capacity = df_airbnb['capacity'].values
minstay = df_airbnb['minstay'].values
ab_rating = df_airbnb['ab_rating'].values
host_rating = df_airbnb['host_rating'].values
coords_airbnb = df_airbnb[['latitude', 'longitude']].to_numpy()

# Extract attractions' coordinates
coords_attractions = df_attractions[['Latitude', 'Longitude']].to_numpy()

# ------------------------
# Tourist Inputs
# ------------------------
print("Please enter your requirements:")
pax = int(input("Number of guests staying: "))
stay_duration = int(input("Number of nights you plan to stay: "))
min_ab_rating = float(input("Minimum Airbnb rating (out of 10): "))
min_host_rating = float(input("Minimum Host rating (out of 10): "))
max_distance = float(input("Maximum distance you're willing to travel to any attraction (in km): "))

print("\nPlease rank the following 5 criteria from 1 (most important) to 5 (least important), separated by commas.")
print("Criteria: Host Rating, Capacity, Stay Duration, Airbnb Rating, Distance to Attractions")
ranking_str = input("Enter your rankings in the format: 2,4,3,5,1\n")
rank_values = list(map(int, ranking_str.strip().split(',')))

criteria_order = ["Superhost", "Capacity", "Stay Duration", "Airbnb Rating", "Distance"]
K = len(criteria_order)
N = len(df_airbnb)

ranks = np.array(rank_values)
inverse_ranks = 1 / ranks
w = inverse_ranks / inverse_ranks.sum()

# ------------------------
# Satisfaction Matrix
# ------------------------
satisfaction_matrix = np.zeros((K, N), dtype=int)
for i in range(N):
    satisfaction_matrix[0][i] = 1 if host_rating[i] >= min_host_rating else 0
    satisfaction_matrix[1][i] = 1 if capacity[i] >= pax else 0
    satisfaction_matrix[2][i] = 1 if stay_duration >= minstay[i] else 0
    satisfaction_matrix[3][i] = 1 if ab_rating[i] >= min_ab_rating else 0

# Compute distances (Airbnb to all attractions)
distance_matrix = cdist(coords_airbnb, coords_attractions, metric='euclidean') * 111  # rough conversion to km
min_distance_to_attraction = distance_matrix.min(axis=1)

# Criterion 5: Distance to nearest attraction within threshold
for i in range(N):
    satisfaction_matrix[4][i] = 1 if min_distance_to_attraction[i] <= max_distance else 0

# ------------------------
# Gurobi Model
# ------------------------
m = gp.Model("Airbnb_Satisfaction_Maximization")
m.setParam('OutputFlag', 0)

y = m.addVars(N, vtype=GRB.BINARY, name="y")
m.setObjective(
    quicksum(y[i] * quicksum(w[k] * satisfaction_matrix[k][i] for k in range(K)) for i in range(N)),
    GRB.MAXIMIZE
)
m.addConstr(quicksum(y[i] for i in range(N)) == 1, name="select_one")

# ------------------------
# Solve
# ------------------------
m.optimize()

# ------------------------
# Results
# ------------------------
if m.status == GRB.OPTIMAL:
    selected = [(i, y[i].x) for i in range(N) if y[i].x > 0.5]
    selected_index = selected[0][0]
    fulfilled_criteria = satisfaction_matrix[:, selected_index].sum()

    df_display = pd.DataFrame(satisfaction_matrix.T, columns=criteria_order)
    df_display.index = airbnb_names
    df_display["Selected"] = ["✅" if int(y[i].x) == 1 else "" for i in range(N)]

    print("\nTourist Inputs:")
    print(f"Pax: {pax}, Stay: {stay_duration}, Min Airbnb Rating: {min_ab_rating}, "
          f"Min Host Rating: {min_host_rating}, Max Travel Distance: {max_distance} km\n")

    print("Optimal Airbnb Selected:")
    print(f"{airbnb_names[selected_index]} fulfilling {fulfilled_criteria} / {K} criteria\n")

    print("Airbnb Criteria Satisfaction Matrix:")
    print(df_display.to_markdown())
else:
    print("No optimal solution found.")

Please enter your requirements:
Number of guests staying: 1
Number of nights you plan to stay: 1
Minimum Airbnb rating (out of 10): 1
Minimum Host rating (out of 10): 1
Maximum distance you're willing to travel to any attraction (in km): 1

Please rank the following 5 criteria from 1 (most important) to 5 (least important), separated by commas.
Criteria: Host Rating, Capacity, Stay Duration, Airbnb Rating, Distance to Attractions
Enter your rankings in the format: 2,4,3,5,1
3,2,5,4,1

Tourist Inputs:
Pax: 1, Stay: 1, Min Airbnb Rating: 1.0, Min Host Rating: 1.0, Max Travel Distance: 1.0 km

Optimal Airbnb Selected:
Studio with private bathroom in the centre 1 fulfilling 5 / 5 criteria

Airbnb Criteria Satisfaction Matrix:
|                                                       |   Superhost |   Capacity |   Stay Duration |   Airbnb Rating |   Distance | Selected   |
|:------------------------------------------------------|------------:|-----------:|----------------:|----------------:|-